# Cleaning the Juliet datasets for near-duplicate functions. 

In [1]:
import os
import sqlite3
import pandas as pd

db_dir = 'datasets'
db_files = [f for f in os.listdir(db_dir) if f.endswith('.db')]
print('Found DB files:', db_files)

schemas = {}
for db_file in db_files:
    db_path = os.path.join(db_dir, db_file)
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f'\nSchema for {db_file}:')
    for table in tables:
        table_name = table[0]
        cursor.execute(f'PRAGMA table_info({table_name});')
        columns = cursor.fetchall()
        print(f'  Table: {table_name}')
        for col in columns:
            print(f'    {col}')
    conn.close()

Found DB files: ['juliet_csharp.db', 'bugsinpy.db', 'juliet_c.db', 'devign.db', 'juliet_java.db']

Schema for juliet_csharp.db:
  Table: funcs
    (0, 'grp', 'TEXT', 0, None, 0)
    (1, 'id', 'TEXT', 0, None, 0)
    (2, 'start', 'INT', 0, None, 0)
    (3, 'end', 'INT', 0, None, 0)
    (4, 'vuln', 'TEXT', 0, None, 0)
    (5, 'code', 'TEXT', 0, None, 0)
    (6, 'len', 'INT', 0, None, 0)

Schema for bugsinpy.db:
  Table: funcs
    (0, 'grp', 'TEXT', 0, None, 0)
    (1, 'id', 'TEXT', 0, None, 0)
    (2, 'start', 'INT', 0, None, 0)
    (3, 'end', 'INT', 0, None, 0)
    (4, 'vuln', 'TEXT', 0, None, 0)
    (5, 'code', 'TEXT', 0, None, 0)
    (6, 'len', 'INT', 0, None, 0)

Schema for juliet_c.db:
  Table: funcs
    (0, 'grp', 'TEXT', 0, None, 0)
    (1, 'id', 'TEXT', 0, None, 0)
    (2, 'start', 'INT', 0, None, 0)
    (3, 'end', 'INT', 0, None, 0)
    (4, 'vuln', 'TEXT', 0, None, 0)
    (5, 'code', 'TEXT', 0, None, 0)
    (6, 'len', 'INT', 0, None, 0)

Schema for devign.db:
  Table: funcs
    

In [12]:
# Query records #73 and #74 from juliet_c and compute SimHash similarities for the first 100 code entries
import os
import re
import sqlite3
import hashlib
from collections import Counter
import pandas as pd
from IPython.display import display

# Locate the juliet_c database
db_path = os.path.join('datasets', 'juliet_c.db')
assert os.path.exists(db_path), f"Database not found at {db_path}"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Find the first table that contains a 'code' column
code_table = None
code_col = 'code'
for (tbl_name,) in cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall():
    cols = [row[1] for row in cursor.execute(f"PRAGMA table_info({tbl_name});").fetchall()]
    if code_col in cols:
        code_table = tbl_name
        break

if not code_table:
    conn.close()
    raise RuntimeError("Couldn't find a table with a 'code' column in juliet_c.db")

print(f"Using table: {code_table}, column: {code_col}")

# Helper: safe SQL identifier quoting (very simple since we got names from PRAGMA)
def ident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

# 1) Query records #73 and #74 (1-based), using rowid ordering
query_73_74 = (
    f"SELECT rowid AS rid, {ident(code_col)} AS code FROM {ident(code_table)} "
    f"WHERE {ident(code_col)} IS NOT NULL ORDER BY rowid LIMIT 2 OFFSET 72;"
)
df_73_74 = pd.read_sql_query(query_73_74, conn)
print("Rows 73 and 74 (by rowid order):")
for _, r in df_73_74.iterrows():
    snippet = (r['code'][:500] + '...') if isinstance(r['code'], str) and len(r['code']) > 500 else r['code']
    print(f"rid={r['rid']}, code length={len(r['code']) if isinstance(r['code'], str) else 'NA'}\n{snippet}\n{'-'*80}")

# 2) Implement SimHash and compute similarities for the first 100 records

def tokenize_code(code: str):
    # Tokenize into identifiers, numbers, multi-char ops, and single symbols
    tokens = re.findall(r"[A-Za-z_]\w+|\d+|==|!=|<=|>=|&&|\|\||[{}()\[\].,;:+\-*/%&|^!<>?=]", code)
    return tokens


def md5_64(text: str) -> int:
    h = hashlib.md5(text.encode('utf-8')).digest()
    # Take high 8 bytes for a 64-bit hash
    return int.from_bytes(h[:8], 'big', signed=False)


def simhash(tokens, hashbits: int = 64) -> int:
    v = [0] * hashbits
    counts = Counter(tokens)
    for token, weight in counts.items():
        h = md5_64(token)
        for i in range(hashbits):
            if h & (1 << i):
                v[i] += weight
            else:
                v[i] -= weight
    fp = 0
    for i in range(hashbits):
        if v[i] >= 0:
            fp |= (1 << i)
    return fp


def hamming_distance(a: int, b: int) -> int:
    return (a ^ b).bit_count()

# Fetch first 100 code entries
query_first_100 = (
    f"SELECT rowid AS rid, {ident(code_col)} AS code FROM {ident(code_table)} "
    f"WHERE {ident(code_col)} IS NOT NULL ORDER BY rowid LIMIT 1000;"
)
codes_df = pd.read_sql_query(query_first_100, conn)
conn.close()

# Compute tokens and SimHash
codes_df['tokens'] = codes_df['code'].astype(str).apply(tokenize_code)
codes_df['simhash'] = codes_df['tokens'].apply(simhash)

# Build pairwise near-duplicate report among the first 100 (4950 pairs)
rids = codes_df['rid'].tolist()
hashes = codes_df['simhash'].tolist()

pairs = []
threshold = 5  # Hamming distance <= 5 is quite similar; adjust as needed
for i in range(len(hashes)):
    hi = hashes[i]
    for j in range(i + 1, len(hashes)):
        d = hamming_distance(hi, hashes[j])
        if d <= threshold:
            pairs.append({
                'rid_i': rids[i],
                'rid_j': rids[j],
                'hamming': d,
                'similarity': 1 - d / 64.0,
            })

pairs_df = pd.DataFrame(pairs).sort_values(['hamming', 'rid_i', 'rid_j']).reset_index(drop=True)
print(f"Near-duplicate pairs among first 1000 rows (Hamming <= {threshold}): {len(pairs_df)} found")
display(pairs_df.head(20))

# If rows 73 and 74 are within the first 100 window, show their distance explicitly
if len(df_73_74) == 2:
    rid73, rid74 = int(df_73_74.iloc[0]['rid']), int(df_73_74.iloc[1]['rid'])
    if rid73 in rids and rid74 in rids:
        i = rids.index(rid73)
        j = rids.index(rid74)
        d = hamming_distance(hashes[i], hashes[j])
        print(f"Hamming distance between row {rid73} and {rid74}: {d} (similarity={1 - d/64.0:.3f})")
else:
    print("Could not fetch both rows 73 and 74.")

# --- Additional metrics: exact duplicates and near-duplicate segment counts ---
# Exact duplicates by exact code text equality
codes_df['code_md5'] = codes_df['code'].astype(str).apply(lambda s: hashlib.md5(s.encode('utf-8')).hexdigest())
md5_counts = codes_df.groupby('code_md5').size().reset_index(name='count')
exact_groups = md5_counts[md5_counts['count'] > 1]
num_exact_groups = int(len(exact_groups))
num_exact_segments = int(codes_df['code_md5'].isin(exact_groups['code_md5']).sum())
num_exact_pairs = int(sum(n * (n - 1) // 2 for n in exact_groups['count']))

# Near-duplicate segments (unique records that appear in at least one near-duplicate pair)
near_dup_segments = set()
for p in pairs:
    near_dup_segments.add(p['rid_i'])
    near_dup_segments.add(p['rid_j'])
num_near_dup_segments = len(near_dup_segments)

# Cluster near-duplicates using Union-Find (to count groups of similar records)
parent = {rid: rid for rid in rids}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for p in pairs:
    union(p['rid_i'], p['rid_j'])

from collections import Counter as Ctr
roots = [find(r) for r in rids]
comp_sizes = Ctr(roots)
cluster_sizes = list(comp_sizes.values())
near_dup_cluster_sizes = [s for s in cluster_sizes if s > 1]
num_near_dup_clusters = len(near_dup_cluster_sizes)

print("\nSummary (first 100 records):")
print(f"- Records examined: {len(codes_df)}")
print(f"- Exact duplicates: {num_exact_segments} segments across {num_exact_groups} groups; {num_exact_pairs} exact duplicate pairs")
print(f"- Near-duplicate pairs (<= {threshold}): {len(pairs_df)}")
print(f"- Near-duplicate segments: {num_near_dup_segments} unique segments involved")
print(f"- Near-duplicate clusters: {num_near_dup_clusters} groups (sizes: {sorted(near_dup_cluster_sizes)[:10]}{'...' if len(near_dup_cluster_sizes) > 10 else ''})")

# --- Verification views: show which records and their snippets ---
# Build a lookup for quick snippet rendering
rid_to_code = dict(zip(codes_df['rid'], codes_df['code']))

def snippet(s: str, n: int = 220) -> str:
    if not isinstance(s, str):
        return ''
    s = s.strip().replace('\n', ' ')
    return (s[:n] + '...') if len(s) > n else s

# Preview first N near-duplicate pairs with code snippets
N = 1264
preview = pairs_df.head(N).copy()
#preview.to_csv('output.csv', index=False)  # Save to CSV if needed

preview['code_i_snippet'] = preview['rid_i'].map(lambda r: snippet(rid_to_code.get(r, '')))
preview['code_j_snippet'] = preview['rid_j'].map(lambda r: snippet(rid_to_code.get(r, '')))
print(f"\nNear-duplicate preview (first {N} pairs):")
display(preview[['rid_i','rid_j','hamming','similarity','code_i_snippet','code_j_snippet']])

# Show members of the largest clusters (by size) for verification
largest_k = 2
# Build cluster membership mapping
from collections import defaultdict
cluster_members = defaultdict(list)
for rid in rids:
    cluster_members[find(rid)].append(rid)
# Sort clusters by size desc
clusters_sorted = sorted(cluster_members.items(), key=lambda kv: len(kv[1]), reverse=True)
print(f"\nTop {largest_k} clusters by size (showing rids):")
for idx, (root, members) in enumerate(clusters_sorted[:largest_k], 1):
    print(f"Cluster {idx}: size={len(members)} rids={sorted(members)[:50]}{'...' if len(members) > 50 else ''}")

Using table: funcs, column: code
Rows 73 and 74 (by rowid order):
rid=73, code length=466
int main(int argc, char * argv[])
{
                         
    srand( (unsigned)time(NULL) );
#ifndef OMITGOOD
    printLine("Calling good()...");
    CWE114_Process_Control__w32_char_connect_socket_21_good();
    printLine("Finished good()");
#endif               
#ifndef OMITBAD
    printLine("Calling bad()...");
    CWE114_Process_Control__w32_char_connect_socket_21_bad();
    printLine("Finished bad()");
#endif              
    return 0;
}
--------------------------------------------------------------------------------
rid=74, code length=466
int main(int argc, char * argv[])
{
                         
    srand( (unsigned)time(NULL) );
#ifndef OMITGOOD
    printLine("Calling good()...");
    CWE114_Process_Control__w32_char_connect_socket_22_good();
    printLine("Finished good()");
#endif               
#ifndef OMITBAD
    printLine("Calling bad()...");
    CWE114_Process_Control__w32_c

,rid_i,rid_j,hamming,similarity
0,1,47,0,1.0
1,1,86,0,1.0
2,1,188,0,1.0
3,2,17,0,1.0
4,2,23,0,1.0
5,2,152,0,1.0
6,2,260,0,1.0
7,2,268,0,1.0
8,2,349,0,1.0
9,2,519,0,1.0


Hamming distance between row 73 and 74: 1 (similarity=0.984)

Summary (first 100 records):
- Records examined: 1000
- Exact duplicates: 318 segments across 53 groups; 1718 exact duplicate pairs
- Near-duplicate pairs (<= 5): 95770
- Near-duplicate segments: 995 unique segments involved
- Near-duplicate clusters: 7 groups (sizes: [5, 5, 8, 8, 40, 238, 691])

Near-duplicate preview (first 1264 pairs):


,rid_i,rid_j,hamming,similarity,code_i_snippet,code_j_snippet
0,1,47,0,1.0,void CWE114_Process_Control__w32_char_connect_...,void CWE114_Process_Control__w32_char_connect_...
1,1,86,0,1.0,void CWE114_Process_Control__w32_char_connect_...,void bad()\r {\r char * data;\r char *...
2,1,188,0,1.0,void CWE114_Process_Control__w32_char_connect_...,void bad()\r {\r char * data;\r char d...
3,2,17,0,1.0,"int main(int argc, char * argv[])\r {\r ...","int main(int argc, char * argv[])\r {\r ..."
4,2,23,0,1.0,"int main(int argc, char * argv[])\r {\r ...","int main(int argc, char * argv[])\r {\r ..."
...,...,...,...,...,...,...
1259,61,409,0,1.0,"int main(int argc, char * argv[])\r {\r ...","int main(int argc, char * argv[])\r {\r ..."
1260,61,415,0,1.0,"int main(int argc, char * argv[])\r {\r ...","int main(int argc, char * argv[])\r {\r ..."
1261,61,428,0,1.0,"int main(int argc, char * argv[])\r {\r ...","int main(int argc, char * argv[])\r {\r ..."
1262,61,441,0,1.0,"int main(int argc, char * argv[])\r {\r ...","int main(int argc, char * argv[])\r {\r ..."



Top 2 clusters by size (showing rids):
Cluster 1: size=691 rids=[1, 3, 5, 6, 7, 8, 9, 11, 12, 13, 14, 16, 18, 19, 20, 21, 22, 24, 25, 27, 28, 30, 31, 32, 33, 34, 36, 37, 39, 40, 42, 43, 44, 45, 47, 48, 49, 51, 53, 54, 55, 56, 57, 59, 60, 62, 64, 65, 66, 68]...
Cluster 2: size=238 rids=[2, 4, 10, 15, 17, 23, 26, 29, 35, 38, 41, 46, 50, 52, 58, 61, 63, 67, 73, 74, 82, 84, 87, 90, 94, 97, 103, 108, 113, 115, 119, 123, 127, 131, 135, 139, 143, 147, 152, 157, 162, 168, 172, 177, 183, 190, 194, 199, 204, 207]...


In [15]:
# --- Minimize removals so all remaining pairs have similarity <= 0.90 ---
# Uses pairs_df and rids computed above.

import math
from collections import defaultdict

sim_threshold = 0.90  # target: all remaining pairs must have similarity <= 0.90

if 'pairs_df' not in globals() or pairs_df is None or pairs_df.empty:
    print("pairs_df is empty or undefined; run the previous cell first.")
else:
    # Build the set of violating edges: pairs with similarity > 0.90
    viol = pairs_df[pairs_df['similarity'] > sim_threshold].copy()

    # If upstream Hamming cutoff was too strict, we may be missing some >0.90 pairs
    if 'hamming' in pairs_df.columns:
        max_h = int(pairs_df['hamming'].max()) if not pairs_df.empty else -1
        # 0.90 corresponds to Hamming <= 6 (since 1 - 6/64 = 0.90625 > 0.90)
        if max_h < 6:
            print("Note: pairs_df was built with a Hamming cutoff < 6; results may be conservative for the 0.90 target.")

    # Edge representation: undirected edge (u, v) with u < v
    def edge_key(a, b):
        return (a, b) if a < b else (b, a)

    # Build edge -> similarity mapping and adjacency
    edges_sim = {}
    adj = defaultdict(set)
    for _, row in viol.iterrows():
        u, v, s = int(row['rid_i']), int(row['rid_j']), float(row['similarity'])
        e = edge_key(u, v)
        if e not in edges_sim or s > edges_sim[e]:  # keep the strongest similarity if duplicates
            edges_sim[e] = s
        adj[u].add(v)
        adj[v].add(u)

    if not edges_sim:
        print("No removals needed; all pairs already <= 0.90.")
    else:
        # Helper to run a greedy cover based on a scoring function for vertices
        def greedy_vertex_cover(edges_sim_map, adj_map, score_fn):
            edges_sim_local = dict(edges_sim_map)
            adj_local = {n: set(neigh) for n, neigh in adj_map.items()}
            removed = []  # vertex removal order

            # Track uncovered edges count quickly
            total_edges = len(edges_sim_local)

            # Precompute incident edges for each node to support fast score updates
            node_to_edges = defaultdict(set)
            for (u,v), s in edges_sim_local.items():
                node_to_edges[u].add((u,v))
                node_to_edges[v].add((u,v))

            # Active nodes (those with any incident violating edges)
            active_nodes = set(n for n, neigh in adj_local.items() if neigh)

            while edges_sim_local:
                # Compute scores only for active nodes
                best_node = None
                best_score = -math.inf
                for n in active_nodes:
                    if not adj_local.get(n):
                        continue
                    sc = score_fn(n, adj_local, edges_sim_local, node_to_edges)
                    if sc > best_score:
                        best_score = sc
                        best_node = n

                if best_node is None:
                    break  # no more edges

                removed.append(best_node)

                # Remove all incident edges of best_node
                for m in list(adj_local.get(best_node, [])):
                    e = edge_key(best_node, m)
                    if e in edges_sim_local:
                        del edges_sim_local[e]
                        if m in adj_local:
                            adj_local[m].discard(best_node)
                    # also clean node_to_edges entries
                    node_to_edges[m].discard(edge_key(min(best_node, m), max(best_node, m)))

                # Clear this node's adjacency
                adj_local[best_node].clear()
                node_to_edges[best_node].clear()

                # Refresh active nodes set (keep only nodes still having incident edges)
                active_nodes = set(n for n, neigh in adj_local.items() if neigh)

            return removed

        # Scoring functions
        def degree_score(n, adj_local, edges_sim_local, node_to_edges):
            # Number of violating edges incident to n
            return len(adj_local.get(n, ()))

        def weighted_score(n, adj_local, edges_sim_local, node_to_edges):
            # Sum of similarities on violating edges incident to n (higher impact first)
            return sum(edges_sim_local[e] for e in node_to_edges.get(n, ()))

        # Run both heuristics and choose the smaller cover
        removed_deg = greedy_vertex_cover(edges_sim, adj, degree_score)
        removed_wgt = greedy_vertex_cover(edges_sim, adj, weighted_score)

        removed_set = set(removed_deg) if len(removed_deg) <= len(removed_wgt) else set(removed_wgt)
        strategy = 'degree' if len(removed_deg) <= len(removed_wgt) else 'weighted'

        # Verify: filter pairs_df to those not involving removed_set, compute max similarity
        remaining_pairs = pairs_df[~pairs_df['rid_i'].isin(removed_set) & ~pairs_df['rid_j'].isin(removed_set)]
        max_sim = float(remaining_pairs['similarity'].max()) if not remaining_pairs.empty else 0.0

        print(f"Strategy chosen: {strategy}")
        print(f"Records to remove so all pairs have similarity <= {sim_threshold:.2f}: {len(removed_set)}")
        print("Removed record IDs:", sorted(removed_set)[:100], ("..." if len(removed_set) > 100 else ""))
        print(f"Remaining max similarity: {max_sim:.6f}")
        if max_sim > sim_threshold:
            print("Note: Remaining pairs with similarity > threshold may exist if they were not included in pairs_df.")

Note: pairs_df was built with a Hamming cutoff < 6; results may be conservative for the 0.90 target.
Strategy chosen: weighted
Records to remove so all pairs have similarity <= 0.90: 945
Removed record IDs: [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101] ...
Remaining max similarity: 0.000000
